# BFSI Lakehouse — Window Functions & Dedup Practice Drills

**Owner:** Rohan Mukherjee  
**Purpose:** Interview-ready practice on window functions, dedup, QUALIFY, and broadcast joins — all on your own `bfsi_lakehouse` tables.

**How to use this notebook:**
- Each item has 3 parts: **Concept** (short) → **Worked Example** (solved, run it) → **Your Turn** (question, you write the code).
- Attempt *Your Turn* before looking anything up. If stuck, revisit the example above it.
- Both **Spark SQL** and **PySpark DataFrame API** are shown, because Databricks interviews test both.

**Tables used** (`bfsi_lakehouse.silver`):

| table | ~rows | key cols |
|---|---|---|
| t_Loan | 3.2M | LoanID, AccountID, LoanSeries, DisbursedAmount, OurBranchID, CreatedAt, LastUpdatedAt |
| t_LoanInstallment | 112M | LoanID, InstallmentSeq, DueAmount, DueDate |
| t_AccountTrx | 20M | TrxID, AccountID, LoanID, TrxAmount, TrxDate |
| t_Client | 1.6M | ClientID, OurBranchID |

> Adjust column names to your actual Silver schema if they differ slightly.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

CATALOG = 'bfsi_lakehouse'
SILVER  = f'{CATALOG}.silver'

# Quick sanity check
spark.sql(f'SELECT count(*) AS n FROM {SILVER}.t_loan').show()


---
## 1. ROW_NUMBER vs RANK vs DENSE_RANK

**The one-liner to remember:**
- `ROW_NUMBER` → 1,2,3,4 — strictly unique, no ties. **Use for dedup** (guarantees exactly one row per key).
- `RANK` → 1,1,3,4 — ties share a number, then a **gap**. Use for 'olympic ranking'.
- `DENSE_RANK` → 1,1,2,3 — ties share, **no gap**. Use for 'top N distinct values'.

**Interview trap:** using `RANK` for dedup. On a tie it keeps *both* rows → duplicate survives.


**Worked example** — show all three side by side on `t_Loan`, partitioned by `LoanID`:


In [0]:
spark.sql(f'''
    SELECT
        LoanID,
        LastUpdatedAt,
        ROW_NUMBER()  OVER (PARTITION BY LoanID ORDER BY LastUpdatedAt DESC) AS rn,
        RANK()        OVER (PARTITION BY LoanID ORDER BY LastUpdatedAt DESC) AS rnk,
        DENSE_RANK()  OVER (PARTITION BY LoanID ORDER BY LastUpdatedAt DESC) AS drnk
    FROM {SILVER}.t_loan
    ORDER BY LoanID, rn
    LIMIT 50
''').show(truncate=False)


**Your Turn (Q1):**  
Across the whole `t_Loan` table, rank branches (`OurBranchID`) by their **total DisbursedAmount** — highest first.  
You want branches with the *same* total to get the *same* rank, and you do **not** want gaps in the ranking numbers.  
Which of the three functions do you use? Write the SQL.


In [0]:
# Q1 — your answer here



---
## 2. QUALIFY — dedup without CTE / subquery

`QUALIFY` filters on a **window function result directly** — no CTE, no wrapping subquery, no outer `WHERE rn = 1`.  
Databricks SQL supports it. It's a strong 'knows-Databricks-SQL' signal.

**Old way (works everywhere):**
```sql
WITH ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY LoanID ORDER BY LastUpdatedAt DESC, CreatedAt DESC) AS rn
  FROM t_loan
)
SELECT * FROM ranked WHERE rn = 1;
```


**Worked example** — same dedup, QUALIFY version (keep latest row per LoanID):


In [0]:
spark.sql(f'''
    SELECT *
    FROM {SILVER}.t_loan
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY LoanID
        ORDER BY LastUpdatedAt DESC, CreatedAt DESC
    ) = 1
''').count()


**Your Turn (Q2):**  
From `t_AccountTrx`, keep only the **latest transaction per AccountID** (by `TrxDate`, tie-break by higher `TrxID`).  
Write it using `QUALIFY` — no CTE allowed.


In [0]:
# Q2 — your answer here



---
## 3. Window dedup in PySpark  (and why NOT `dropDuplicates`)

`df.dropDuplicates(['LoanID'])` keeps an **arbitrary** row per key — you don't control which one. For 'keep latest' this is **wrong**; it may keep the stale row.

Correct pattern = `Window.partitionBy(keys).orderBy(col.desc())` + `row_number() == 1`. This is exactly what your `dedup_for_merge()` does.

**When `dropDuplicates` is fine:** rows are fully identical and you just want distinct, and you don't care which copy survives.


**Worked example** — DataFrame-API dedup, keep latest per LoanID:


In [0]:
df = spark.table(f'{SILVER}.t_loan')

w = Window.partitionBy('LoanID').orderBy(
        F.col('LastUpdatedAt').desc_nulls_last(),
        F.col('CreatedAt').desc_nulls_last())

df_dedup = (df.withColumn('_rn', F.row_number().over(w))
              .filter(F.col('_rn') == 1)
              .drop('_rn'))

print('before:', df.count(), '| after:', df_dedup.count())


**Your Turn (Q3):**  
In one line, explain what `df.dropDuplicates(['LoanID'])` returns for a LoanID that appears 3 times with different `LastUpdatedAt`.  
Then write the correct DataFrame-API code to keep the **earliest** (oldest) row per LoanID instead of the latest.


In [0]:
# Q3 — your explanation (comment) + code here



---
## 4. Top-N per group

Classic interview question: 'top 3 X per Y'. Always `ROW_NUMBER` (or `RANK` if ties should be included) partitioned by the group, filtered to `<= N`.


**Worked example** — top 3 largest installments per LoanID:


In [0]:
spark.sql(f'''
    SELECT LoanID, InstallmentSeq, DueAmount
    FROM {SILVER}.t_loaninstallment
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY LoanID ORDER BY DueAmount DESC
    ) <= 3
''').show(20, truncate=False)


**Your Turn (Q4):**  
For each `OurBranchID`, find the **top 5 loans by DisbursedAmount**. Return branch, LoanID, amount.  
Do it once in **Spark SQL** (QUALIFY) and once in **PySpark DataFrame API** (Window + filter).


In [0]:
# Q4 — SQL version



In [0]:
# Q4 — DataFrame API version



---
## 5. LAG / LEAD — compare a row to its neighbour

`LAG(col, 1)` = previous row's value in the ordered window. `LEAD` = next row.  
Use for: 'change since last transaction', 'days between installments', 'did balance drop'.


**Worked example** — for each account, amount change vs its previous transaction:


In [0]:
spark.sql(f'''
    SELECT
        AccountID, TrxID, TrxDate, TrxAmount,
        LAG(TrxAmount) OVER (PARTITION BY AccountID ORDER BY TrxDate, TrxID) AS prev_amt,
        TrxAmount - LAG(TrxAmount) OVER (PARTITION BY AccountID ORDER BY TrxDate, TrxID) AS delta
    FROM {SILVER}.t_accounttrx
    LIMIT 50
''').show(truncate=False)


**Your Turn (Q5):**  
For each `LoanID` in `t_LoanInstallment` (ordered by `InstallmentSeq`), compute the **gap in days between each installment's DueDate and the previous one**.  
First installment of each loan should show NULL. Write the SQL.


In [0]:
# Q5 — your answer here



---
## 6. Running / cumulative total (window frame)

A cumulative sum needs an explicit **frame**: `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`.  
Without a frame, `SUM() OVER (... ORDER BY ...)` defaults to `RANGE UNBOUNDED PRECEDING` — which can behave unexpectedly on ties. Interviewers love asking about this default.


**Worked example** — running total of transaction amount per account:


In [0]:
spark.sql(f'''
    SELECT
        AccountID, TrxDate, TrxAmount,
        SUM(TrxAmount) OVER (
            PARTITION BY AccountID
            ORDER BY TrxDate, TrxID
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_balance
    FROM {SILVER}.t_accounttrx
    LIMIT 50
''').show(truncate=False)


**Your Turn (Q6):**  
For each `LoanID`, compute the **cumulative sum of DueAmount** across installments ordered by `InstallmentSeq` (i.e. how much is due up to and including each installment).  
Write the SQL with an explicit frame. Bonus: what does the frame default to if you omit `ROWS BETWEEN ...`?


In [0]:
# Q6 — your answer + bonus (comment) here



---
## 7. Broadcast join — kill the shuffle

When one side of a join is **small** (rule of thumb < ~10–100MB, default `autoBroadcastJoinThreshold` = 10MB), broadcast it. Spark ships the small table to every executor, so the **big table never shuffles**. This turns a shuffle-hash / sort-merge join into a broadcast-hash join.

**Why it helps your case:** joining `t_LoanInstallment` (112M) to `t_Loan` (3.2M) on `LoanID`. Even 3.2M might be too big to auto-broadcast, but a *slim projection* of t_Loan (just LoanID + 2 cols) can be small enough.

**Verify in `.explain()`** — look for `BroadcastHashJoin`, not `SortMergeJoin`.


**Worked example** — broadcast the small side, then confirm via explain:


In [0]:
big   = spark.table(f'{SILVER}.t_loaninstallment')
small = (spark.table(f'{SILVER}.t_loan')
              .select('LoanID', 'DisbursedAmount', 'OurBranchID'))  # slim projection

joined = big.join(F.broadcast(small), on='LoanID', how='left')

joined.explain(mode='formatted')   # look for BroadcastHashJoin


**Your Turn (Q7):**  
1. Write a broadcast join of `t_AccountTrx` (20M) enriched with `t_Client` (1.6M) info via `OurBranchID` (assume both have it).  
2. Then answer in a comment: *why can broadcasting a table that is actually too large blow up your driver / executors?* (This is the standard follow-up.)


In [0]:
# Q7 — join code + explanation comment here



---
## 8. (Stretch) Gap-and-island

Harder pattern that shows up in senior interviews: find **continuous streaks** or **breaks** in a sequence. Trick = `row_number()` difference, or `LAG` + running sum of a 'new group' flag.

Example use on your data: find runs of **consecutive on-time installments** per loan, or detect missing `InstallmentSeq` gaps.


**Worked example** — detect missing InstallmentSeq gaps per LoanID (a seq is missing if prev+1 != current):


In [0]:
spark.sql(f'''
    WITH seq AS (
        SELECT LoanID, InstallmentSeq,
               LAG(InstallmentSeq) OVER (PARTITION BY LoanID ORDER BY InstallmentSeq) AS prev_seq
        FROM {SILVER}.t_loaninstallment
    )
    SELECT LoanID, prev_seq, InstallmentSeq,
           (InstallmentSeq - prev_seq) AS gap_size
    FROM seq
    WHERE prev_seq IS NOT NULL
      AND InstallmentSeq <> prev_seq + 1
    LIMIT 50
''').show(truncate=False)


**Your Turn (Q8):**  
Using `t_AccountTrx`, for each `AccountID` assign a **group id** that increments every time the gap between consecutive `TrxDate` values is more than 30 days.  
(Classic 'sessionization' / island problem. Hint: build a 0/1 'new island' flag with LAG, then a running SUM of that flag.)  
Write the SQL. This is the hardest one — take your time.


In [0]:
# Q8 — your answer here



---
## Answer key policy

No answers included on purpose — attempt each, then paste your code back into chat and I'll review like an interviewer (correctness + the follow-up they'd ask).

**Priority order for you:** Q2 (QUALIFY), Q7 (broadcast) first — those came up live today and are your weak spots. Then Q5/Q6 (LAG + frames), then Q8 (islands) as a stretch.
